# Indexes, EXPLAIN and Query Optimization

Notebook for **Chapter 10** of the *sql* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/sql/ch10.html

Tip: change a number, predict the output, then run the cell again.

## 10.1 What an Index Is, and What It Costs

## 10.2 Composite, Covering, Partial and Expression Indexes

## 10.3 Reading EXPLAIN and EXPLAIN ANALYZE

## 10.4 Query Optimization: A Measure-First Method

## 10.5 Performance Lab: Sargability, Pagination and Statistics You Can Measure

In [ ]:
import sqlite3, random
random.seed(7)
db = sqlite3.connect(":memory:", isolation_level=None)
db.execute("CREATE TABLE big (id INTEGER PRIMARY KEY, customer_id INT, created TEXT, email TEXT, amount INT)")
rows = [(i, random.randint(1, 2000), "2024-%02d-%02d" % (random.randint(1, 12), random.randint(1, 28)),
         "user%d@Mail.com" % i, random.randint(10, 999)) for i in range(1, 50001)]
db.executemany("INSERT INTO big VALUES (?,?,?,?,?)", rows)

def cost(q, args=()):
    """count SQLite virtual-machine steps: a stable stand-in for 'work done'"""
    n = [0]
    def tick():
        n[0] += 1
        return 0
    db.set_progress_handler(tick, 1)
    db.execute(q, args).fetchall()
    db.set_progress_handler(None, 0)
    return n[0]

def plan(q):
    return db.execute("EXPLAIN QUERY PLAN " + q).fetchall()[0][3]

tests = [
  ("function on column", "SELECT count(*) FROM big WHERE substr(created, 1, 7) = '2024-03'",
                         "SELECT count(*) FROM big WHERE created >= '2024-03-01' AND created < '2024-04-01'", None),
  ("lower() on column",  "SELECT count(*) FROM big WHERE lower(email) = 'user77@mail.com'",
                         "SELECT count(*) FROM big WHERE lower(email) = 'user77@mail.com'",
                         "CREATE INDEX i_lemail ON big(lower(email))"),
  ("arithmetic on column", "SELECT count(*) FROM big WHERE customer_id + 1 = 43",
                         "SELECT count(*) FROM big WHERE customer_id = 42", None),
]
db.execute("CREATE INDEX i_created ON big(created)")
db.execute("CREATE INDEX i_cust ON big(customer_id)")
for name, bad, good, new_index in tests:
    print(name)
    print("  slow :", cost(bad), "steps |", plan(bad))
    if new_index:
        db.execute(new_index)
        print("  (created the expression index)")
    print("  fast :", cost(good), "steps |", plan(good))

function on column
  slow : 204116 steps | SCAN big USING COVERING INDEX i_created
  fast : 12322 steps | SEARCH big USING COVERING INDEX i_created (created>? AND created<?)
lower() on column
  slow : 200013 steps | SCAN big
  (created the expression index)
  fast : 18 steps | SEARCH big USING COVERING INDEX i_lemail (<expr>=?)
arithmetic on column
  slow : 200031 steps | SCAN big USING COVERING INDEX i_cust
  fast : 64 steps | SEARCH big USING COVERING INDEX i_cust (customer_id=?)


In [ ]:
import sqlite3, random
random.seed(7)
db = sqlite3.connect(":memory:", isolation_level=None)
db.execute("CREATE TABLE big (id INTEGER PRIMARY KEY, customer_id INT, created TEXT, email TEXT, amount INT)")
rows = [(i, random.randint(1, 2000), "2024-%02d-%02d" % (random.randint(1, 12), random.randint(1, 28)),
         "user%d@Mail.com" % i, random.randint(10, 999)) for i in range(1, 50001)]
db.executemany("INSERT INTO big VALUES (?,?,?,?,?)", rows)

def cost(q, args=()):
    """count SQLite virtual-machine steps: a stable stand-in for 'work done'"""
    n = [0]
    def tick():
        n[0] += 1
        return 0
    db.set_progress_handler(tick, 1)
    db.execute(q, args).fetchall()
    db.set_progress_handler(None, 0)
    return n[0]

def plan(q):
    return db.execute("EXPLAIN QUERY PLAN " + q).fetchall()[0][3]

db.execute("CREATE INDEX i_ci ON big(created, id)")
for page in (1, 100, 500):
    off = (page - 1) * 20
    a = cost("SELECT id, created FROM big ORDER BY created, id LIMIT 20 OFFSET ?", (off,))
    last = db.execute("SELECT created, id FROM big ORDER BY created, id LIMIT 1 OFFSET ?", (max(off - 1, 0),)).fetchone()
    k = cost("SELECT id, created FROM big WHERE (created, id) > (?, ?) ORDER BY created, id LIMIT 20", last)
    print("page %3d  OFFSET: %6d steps   keyset: %5d steps" % (page, a, k))

page   1  OFFSET:    130 steps   keyset:   263 steps
page 100  OFFSET:   4090 steps   keyset:   765 steps
page 500  OFFSET:  20090 steps   keyset:   738 steps


In [ ]:
import sqlite3
db = sqlite3.connect(":memory:", isolation_level=None)
db.execute("CREATE TABLE ev (id INTEGER PRIMARY KEY, status TEXT, v INT)")
db.executemany("INSERT INTO ev VALUES (?,?,?)", [(i, 'done' if i % 100 else 'failed', i) for i in range(1, 50001)])
db.execute("CREATE INDEX i_status ON ev(status)")

def run(label):
    for st in ("done", "failed"):
        q = "SELECT count(*), sum(v) FROM ev WHERE status = '%s'" % st
        n = [0]
        def tick():
            n[0] += 1
            return 0
        db.set_progress_handler(tick, 1)
        db.execute(q).fetchall()
        db.set_progress_handler(None, 0)
        p = db.execute("EXPLAIN QUERY PLAN " + q).fetchall()[0][3]
        print(label, st.ljust(6), str(n[0]).rjust(7), "steps |", p)

run("before ANALYZE:")
db.execute("ANALYZE")
print(db.execute("SELECT tbl, idx, stat FROM sqlite_stat1").fetchall())
run("after  ANALYZE:")

before ANALYZE: done    297014 steps | SEARCH ev USING INDEX i_status (status=?)
before ANALYZE: failed    3013 steps | SEARCH ev USING INDEX i_status (status=?)
[('ev', 'i_status', '50000 25000')]
after  ANALYZE: done    298513 steps | SCAN ev
after  ANALYZE: failed  151513 steps | SCAN ev


## 10.6 Index Types, Join Algorithms and Partitioning

In [ ]:
import sqlite3, random
random.seed(7)
db = sqlite3.connect(":memory:", isolation_level=None)
db.execute("CREATE TABLE cust (customer_id INTEGER PRIMARY KEY, name TEXT)")
db.executemany("INSERT INTO cust VALUES (?,?)", [(i, "c%d" % i) for i in range(1, 2001)])
db.execute("CREATE TABLE big (id INTEGER PRIMARY KEY, customer_id INT, created TEXT, email TEXT, amount INT)")
rows = [(i, random.randint(1, 2000), "2024-%02d-%02d" % (random.randint(1, 12), random.randint(1, 28)),
         "user%d@Mail.com" % i, random.randint(10, 999)) for i in range(1, 50001)]
db.executemany("INSERT INTO big VALUES (?,?,?,?,?)", rows)


def cost(q):
    n = [0]
    def tick():
        n[0] += 1
        return 0
    db.set_progress_handler(tick, 1)
    db.execute(q).fetchall()
    db.set_progress_handler(None, 0)
    return n[0]

q = "SELECT count(*) FROM cust c JOIN big b ON b.customer_id = c.customer_id WHERE c.customer_id <= 50"
def plans(q):
    return " ; ".join(r[3] for r in db.execute("EXPLAIN QUERY PLAN " + q).fetchall())
print("no index   :", cost(q), "steps |", plans(q))
db.execute("CREATE INDEX i_cust ON big(customer_id)")
print("with index :", cost(q), "steps |", plans(q))

no index   : 251242 steps | SCAN b ; SEARCH c USING INTEGER PRIMARY KEY (rowid=?)
with index : 8613 steps | SEARCH b USING COVERING INDEX i_cust (customer_id<?) ; SEARCH c USING INTEGER PRIMARY KEY (rowid=?)


## Chapter recap

## Practice set

Problem 10.P.1 (easy) Which is more selective: `WHERE gender = 'F'` or `WHERE email = 'a@x.com'`? Which deserves an index? **Answer:** The email filter (about one row). Index `email`. A two-value column has low selectivity.

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/sql/ch10.html